# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [1]:
%%sh
git config --global user.email "sir.alexandr1998@gmail.com"
git config --global user.name "pankratov.ad"
git init
pip install scikit-learn numpy pandas -qqq
pip freeze > requirements.txt

Initialized empty Git repository in /content/.git/


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


In [2]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Writing ml_pipeline.py


### Проверяем работоспособность пайплайна

In [3]:
!python ml_pipeline.py

Точность аccuracy: 1.00


In [4]:
%%writefile .gitlab-ci.yml
name: Gitea Actions Demo
run-name: ${{ gitea.actor }} is testing out Gitea Actions 🚀
on: [push]

jobs:
  Explore-Gitea-Actions:
    runs-on: ubuntu-latest
    steps:
      - run: echo "🎉 The job was automatically triggered by a ${{ gitea.event_name }} event."
      - run: echo "🐧 This job is now running on a ${{ runner.os }} server hosted by Gitea!"
      - run: echo "🔎 The name of your branch is ${{ gitea.ref }} and your repository is ${{ gitea.repository }}."
      - name: Check out repository code
        uses: actions/checkout@v4
      - run: echo "💡 The ${{ gitea.repository }} repository has been cloned to the runner."
      - run: echo "🖥️ The workflow is now ready to test your code on the runner."
      - name: List files in the repository
        run: |
          ls ${{ gitea.workspace }}
      - run: echo "🍏 This job's status is ${{ job.status }}."


Writing .gitlab-ci.yml


In [5]:
%%writefile .gitverse-ci.yaml
stages:
  - test

ml_service_job:
  stage: test
  # Фиксируем образ для воспроизводимости (вместо runs-on: ubuntu-latest)
  image: python:3.10-slim

  script:
    - echo "🎉 The job was automatically triggered by a push event."
    - echo "🐧 This job is now running on a server hosted by GitVerse!"
    - echo "🔎 The name of your branch is $CI_COMMIT_BRANCH and your repository is $CI_PROJECT_PATH."
    - echo "💡 The repository has been cloned to the runner."
    - echo "🖥️ The workflow is now ready to test your code on the runner."
    - ls -la

    # --- ПЕРЕПИСАННЫЙ ШАГ: Make pipeline reproducible ---
    - echo "🔧 Make pipeline reproducible: Fixing dependencies and environment..."
    - pip install --upgrade pip
    - pip install numpy==1.24.3 pandas==2.0.3 scikit-learn==1.3.1
    - echo "🚀 Running ML pipeline..."
    - python ml_pipeline.py
    # --------------------------------------------------

    - echo "🍏 This job's status is $CI_JOB_STATUS."

Writing .gitverse-ci.yaml


In [6]:
!git add .gitlab-ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitLab"
!git log

[master (root-commit) e3dbeb5] build(ml_pipeline.py) добавлен пайплайн GitLab
 2 files changed, 32 insertions(+)
 create mode 100644 .gitlab-ci.yml
 create mode 100644 ml_pipeline.py
commit e3dbeb5515a1766d28809308bdc8fa8a71e0220e (HEAD -> master)
Author: pankratov.ad <sir.alexandr1998@gmail.com>
Date:   Tue May 12 12:37:28 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitLab


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

```
https://gitverse.ru/pankratov.ad/HW7_CI_CD/content/master/.gitverse/workflows/gitverse-ci.yaml
```

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

```
ADR-0001: Выбор стратегии теневого развертывания (Shadow) для ML-сервиса без обработки ошибок
Дата: 2026-05-10
Статус: принято
Контекст
В текущем коде ML-сервиса (ml_pipeline.py) полностью отсутствует обработка ошибок (блоки try-except, валидация входных данных, проверка на None). При столкновении с некорректными данными в продакшене (например, NaN, неверный размер массива, нехватка памяти) сервис будет падать с критическими ошибками (HTTP 500 или краш контейнера), не выдавая понятных логов. Необходимо выбрать стратегию деплоя, которая позволит проверить стабильность сервиса на реальных данных с минимальным влиянием на пользователей, прежде чем переключать на него весь трафик.

Решение
Мы выбираем стратегию Теневого развертывания (Shadow Deployment).Реальные пользовательские запросы дублируются и отправляются в новую версию ML-сервиса параллельно с текущей (стабильной) версией. Ответы от теневой версии игнорируются и не возвращаются пользователям.

Причины выбора:
Безопасность при отсутствии error handling: Так как код не имеет обработки ошибок, падение сервиса неизбежно при появлении аномалий в данных. Shadow позволяет увидеть эти падения на реальном трафике, не разрушая пользовательский опыт.
Сбор реальных метрик: Мы можем измерить реальную задержку (latency) и потребление памяти моделью на продакшен-данных, не рискуя репутацией.
Время на исправление: Дает команде разработчиков время на добавление блоков try-except и логирования в код, наблюдая за реальными паттернами отказов в теневой среде.
Альтернативы
Альтернатива 1: Blue-Green Deployment
Описание: Поднимается две идентичные среды. Трафик переключается с "синей" (старой) на "зеленую" (новую) мгновенно с помощью изменения конфигурации роутера.Сравнение: В отличие от Shadow, при переключении на "зеленую" среду пользователи сразу начнут получать ошибки. Если код упадет из-за отсутствия try-except, откат займет несколько секунд/минут, за которые часть пользователей получит 500-е ошибки.

Альтернатива 2: Canary Release
Описание: Трафик постепенно перенаправляется на новую версию (например, 5% -> 25% -> 100%).Сравнение: В отличие от Shadow, здесь реальные пользователи получают ответы от нового кода. Из-за отсутствия обработки ошибок, даже 5% пользователей, попавших под "канарейку", столкнутся с падениями сервиса, что недопустимо для критичных ML-инференсов.

Последствия и риски
Положительные последствия:

Нулевой риск для конечных пользователей при тестировании нестабильного кода.
Выявление узких мест и необработанных исключений на реальных продакшен-данных до фактического релиза.
Риски, к которым привело это решение:

Риск "иллюзии безопасности" (False Confidence): Теневой трафик может не покрывать 100% edge-cases. Код может нормально отработать в тени, но упасть при реальном переключении из-за разницы в таймаутах или состоянии БД.
Высокие затраты на инфраструктуру: Требуется в 2 раза больше вычислительных ресурсов (CPU/RAM), так как ML-модель выполняет инференс дважды для каждого запроса, что критично для тяжелых моделей.
Риск затягивания релиза ("Shadow Forever"): Команда может привыкнуть к теневому тестированию и откладывать реальный релиз и, что важнее, рефакторинг кода для добавления обработки ошибок.
Сложность отладки side-effects: Если бы ML-сервис не только читал данные, но и писал их (например, логи предсказаний в БД), теневой деплой мог бы вызвать дублирование записей или ошибки консистентности данных.

```

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [7]:
%%writefile docker-compose-blue.yaml
# Реализация стратегии Теневого развертывания (Shadow Deployment).
# Так как Docker Compose не умеет дублировать трафик "из коробки",
# используется Traefik как точка входа, который обладает встроенной
# функцией mirroring (зеркалирования) трафика.

version: '3.8'

services:
  # Прокси-сервер для маршрутизации и зеркалирования запросов
  traefik:
    image: traefik:v2.10
    command:
      - "--api.insecure=true" # Включаем дашборд для наглядности
      - "--providers.docker=true" # Включаем поддержку меток Docker
      - "--providers.file.filename=/etc/traefik/dynamic.yml" # Подключаем файл с правилами зеркалирования
    ports:
      - "80:80"    # Основной порт для пользователей
      - "8080:8080" # Порт дашборда (опционально)
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock:ro
      # Для работы этого файла нужен отдельный файл конфигурации
      - ./traefik-dynamic.yml:/etc/traefik/dynamic.yml:ro
    networks:
      - shadow_network

  # Стабильная версия сервиса ("Blue" / Продакшен)
  ml-service-blue:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: ml_service_blue
    networks:
      - shadow_network
    # Метки нужны, чтобы Traefik знал, как маршрутизировать основной трафик
    labels:
      - "traefik.enable=true"
      - "traefik.http.routers.blue.rule=PathPrefix(`/`)"
      - "traefik.http.routers.blue.entrypoints=web"
      - "traefik.http.services.blue.loadbalancer.server.port=5000"

  # Теневая версия сервиса (Shadow)
  # Она получает дубликаты запросов, но её ответы НИКОГДА не доходят до пользователей.
  # Если она упадет из-за отсутствия try-except, пользователи этого не заметят.
  ml-service-shadow:
    build:
      context: .
      dockerfile: Dockerfile.shadow
    container_name: ml_service_shadow
    networks:
      - shadow_network
    # У теневого сервиса НЕТ меток роутера, он скрыт от прямого доступа

networks:
  shadow_network:
    driver: bridge

Writing docker-compose-blue.yaml


In [8]:
%%writefile traefik-dynamic.yml
http:
  services:
    blue-with-shadow:
      loadBalancer:
        servers:
          - url: "http://ml-service-blue:5000"
        mirrors:
          - name: shadow
            percent: 100 # Отправляем 100% дубликатов запросов в тень
    shadow:
      loadBalancer:
        servers:
          - url: "http://ml-service-shadow:5000"
  routers:
    main:
      rule: "PathPrefix(`/`)"
      entryPoints:
        - web
      service: blue-with-shadow

Writing traefik-dynamic.yml


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [9]:
from dataclasses import dataclass
from typing import Dict, Tuple
from scipy import stats
import math

@dataclass
class MLABTestPlan:
    """
    Класс для описания параметров A/B-тестирования ML-модели.
    Контекст: Сравнение текущей продакшен-модели (RandomForest)
    с новой версией (например, LogisticRegression или обновленный RandomForest).
    """

    # 1. Общие параметры эксперимента
    experiment_name: str = "ml_model_v2_vs_v1"
    description: str = "Сравнение точности (accuracy) и времени отклика (latency) двух версий ML-сервиса"

    # 2. Группы и распределение трафика
    traffic_split: Dict[str, float] = None
    control_group_name: str = "model_v1_rf"   # Текущая стабильная модель
    treatment_group_name: str = "model_v2_lr"  # Новая тестируемая модель

    # 3. Целевые метрики (Primary & Secondary)
    primary_metric: str = "accuracy"
    secondary_metrics: Tuple[str, ...] = ("p95_latency_ms", "error_rate")

    # 4. Статистические параметры
    baseline_conversion_rate: float = 0.96  # Ожидаемая accuracy модели V1 (из логов/тестов)
    minimum_detectable_effect: float = 0.02 # Мы хотим обнаружить улучшение минимум на 2% (абсолютно)
    statistical_significance_level: float = 0.05  # Альфа (вероятность ложноположительного результата)
    statistical_power: float = 0.80              # Бета (вероятность обнаружить реальный эффект, 1 - beta)

    def __post_init__(self):
        if self.traffic_split is None:
            self.traffic_split = {self.control_group_name: 0.5, self.treatment_group_name: 0.5}

    def calculate_required_sample_size(self) -> int:
        """
        Расчет необходимого размера выборки для каждой группы.
        Используется формула для двухпропорционального z-теста.
        """
        p1 = self.baseline_conversion_rate
        p2 = p1 + self.minimum_detectable_effect

        # Z-критерии для альфа и мощности
        z_alpha = stats.norm.ppf(1 - self.statistical_significance_level / 2)
        z_beta = stats.norm.ppf(self.statistical_power)

        # Средняя пропорция
        p_avg = (p1 + p2) / 2

        # Формула размера выборки для каждой группы
        numerator = (z_alpha * math.sqrt(2 * p_avg * (1 - p_avg)) +
                     z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
        denominator = (p1 - p2) ** 2

        n_per_group = math.ceil(numerator / denominator)
        total_sample = n_per_group * 2

        return total_sample, n_per_group

    def print_plan_summary(self):
        """Вывод итогового плана эксперимента."""
        total, per_group = self.calculate_required_sample_size()

        print("-"*60)
        print(f"ПЛАН A/B-ТЕСТА: {self.experiment_name.upper()}")
        print("-"*60)
        print(f"Описание: {self.description}")
        print("-" * 60)
        print("【 ПАРАМЕТРЫ РАСПРЕДЕЛЕНИЯ ТРАФИКА 】")
        for group, share in self.traffic_split.items():
            role = "Контроль" if group == self.control_group_name else "Тест"
            print(f"  • {role} ({group}): {int(share*100)}%")
        print("-" * 60)
        print("【 ЦЕЛЕВЫЕ МЕТРИКИ 】")
        print(f"  • Основная: {self.primary_metric}")
        print(f"  • Вторичные: {', '.join(self.secondary_metrics)}")
        print("-" * 60)
        print("【 СТАТИСТИЧЕСКИЕ ПАРАМЕТРЫ 】")
        print(f"  • Базовая метрика (Control): {self.baseline_conversion_rate:.2f}")
        print(f"  • Минимально обнаружимый эффект (MDE): +{self.minimum_detectable_effect:.2f}")
        print(f"  • Уровень значимости (α): {self.statistical_significance_level}")
        print(f"  • Мощность теста (1-β): {self.statistical_power}")
        print("-" * 60)
        print("【 ТРЕБОВАНИЯ К ДАННЫМ (РАЗМЕР ВЫБОРКИ) 】")
        print(f"  • Запросов на группу: {per_group:,}")
        print(f"  • ВСЕГО запросов нужно собрать: {total:,}")
        print("-"*60)

# Инициализация и вывод плана
if __name__ == "__main__":
    ab_test = MLABTestPlan()
    ab_test.print_plan_summary()

------------------------------------------------------------
ПЛАН A/B-ТЕСТА: ML_MODEL_V2_VS_V1
------------------------------------------------------------
Описание: Сравнение точности (accuracy) и времени отклика (latency) двух версий ML-сервиса
------------------------------------------------------------
【 ПАРАМЕТРЫ РАСПРЕДЕЛЕНИЯ ТРАФИКА 】
  • Контроль (model_v1_rf): 50%
  • Тест (model_v2_lr): 50%
------------------------------------------------------------
【 ЦЕЛЕВЫЕ МЕТРИКИ 】
  • Основная: accuracy
  • Вторичные: p95_latency_ms, error_rate
------------------------------------------------------------
【 СТАТИСТИЧЕСКИЕ ПАРАМЕТРЫ 】
  • Базовая метрика (Control): 0.96
  • Минимально обнаружимый эффект (MDE): +0.02
  • Уровень значимости (α): 0.05
  • Мощность теста (1-β): 0.8
------------------------------------------------------------
【 ТРЕБОВАНИЯ К ДАННЫМ (РАЗМЕР ВЫБОРКИ) 】
  • Запросов на группу: 1,141
  • ВСЕГО запросов нужно собрать: 2,282
------------------------------------------

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [10]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
iris = load_iris();X = iris.data ;y = iris.target
hyperparameters={"n_estimators":100, "random_state":42}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train);y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Точность аccuracy: {accuracy:.2f}')

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [11]:
!python ml_pipeline.py

Точность аccuracy: 1.00


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [12]:
%%writefile ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2
    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.x'
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install scikit-learn numpy pandas
    - name: Run pipeline
      run: |
        python -c "import numpy as np; import pandas as pd; from sklearn.datasets import load_iris; from sklearn.model_selection import train_test_split; from sklearn.ensemble import RandomForestClassifier; from sklearn.metrics import accuracy_score; iris = load_iris(); X = iris.data; y = iris.target; X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42); model = RandomForestClassifier(n_estimators=100, random_state=42); model.fit(X_train, y_train); y_pred = model.predict(X_test); accuracy = accuracy_score(y_test, y_pred); print(f'Accuracy: {accuracy:.2f}')"
    - name: Make pipeline reproducible
      run: |
        python -c "print('какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн?')"


Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [13]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [14]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[master f4c5a82] build(ml_pipeline.py) добавлен пайплайн GitHub Actions
 1 file changed, 23 insertions(+)
 create mode 100644 .github/workflows/ci.yml
commit f4c5a82c6cd672fd2945d8c9b6bbba1bc0d9207b (HEAD -> master)
Author: pankratov.ad <sir.alexandr1998@gmail.com>
Date:   Tue May 12 12:37:34 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitHub Actions

commit e3dbeb5515a1766d28809308bdc8fa8a71e0220e
Author: pankratov.ad <sir.alexandr1998@gmail.com>
Date:   Tue May 12 12:37:28 2026 +0000

    build(ml_pipeline.py) добавлен пайплайн GitLab


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

```
github:
https://github.com/SunCheze/HW7_CI_CD.git

gitverse:
https://gitverse.ru/pankratov.ad/HW7_CI_CD.git
```

## 6. Итоговое оформление

В процессе освоения модуля самым сложным этапом стало понимание общего алгоритма CI/CD и неоднозначные требования к заданиям, из-за чего архитектура всего процесса поначалу казалась запутанной. Долгое время инструменты (Git, Docker, CI-системы, балансировщики) казались абсолютно разрозненными сущностями, но в итоге удалось осознать их логичную связность: Git контролирует изменения, Docker упаковывает код в изолированную среду, а CI/CD-платформы выступают "кровеносной системой", автоматизируя прогон тестов и передачу артефактов. Проще всего далось написание самого ML-кода и базовая настройка синтаксиса конфигурационных файлов, так как эти процессы имеют четкую структуру. Значительные трудности вызвало сопоставление синтаксисов разных платформ (например, GitLab CI против GitHub/Gitea Actions) и практическая реализация сложных стратегий деплоя через инструменты маршрутизации трафика. Главный вывод по обоснованию стратегий деплоя заключается в том, что выбор подхода не является шаблонным и напрямую диктуется качеством кода: например, отсутствие обработки ошибок делает canary релиз недопустимым риском, вынуждая использовать теневое развертывание ради безопасности реальных пользователей.

